# Step 10 — VDAP DMX to SEISAN WAV

Convert Pinatubo legacy waveforms with FLOVOpy, retaining low-gain IDs such as `PI2L → XB.PI2..ELZ`. Paths match `run_pinatubo_pipeline.sh`. Choose a kernel with the updated FLOVOpy installation and ObsPy.

This notebook writes waveform files and metadata when the conversion cell runs. Existing waveforms are reused by default; set `OVERWRITE=True` to regenerate them with current settings. The 100 Hz correction changes metadata, not samples. The reader subtracts the legacy 2048-count offset.


In [ ]:
from pathlib import Path
from IPython.display import display
from flovopy.core.vdap import convert_dmx_to_seisan

DATA_TOP = Path.home() / "Dropbox/PROFESSIONAL/DATA/Pinatubo/ASSEMBLED-25-028-THOMPSON-PINATUBO1991"
SUDS_TOP = DATA_TOP / "LEGACY/WAVEFORM_DATA/SUDS"
FAIR_TOP = DATA_TOP / "FAIR"
NETWORK = "XB"
DATABASE = "PNTBO"
SEISAN_WAV_DB = FAIR_TOP / "SEISAN/WAV" / DATABASE
FAIR_META_DIR = FAIR_TOP / "metadata"
TMP_DIR = DATA_TOP / "pipeline_tmp"

SAMPLING_RATE = 100.0
PATTERN = "**/*.DMX"
MAX_FILES = None  # Set to a small positive integer for a trial run.
OVERWRITE = False
VERBOSE = False


In [ ]:
if not SUDS_TOP.is_dir():
    raise FileNotFoundError(f"Check DATA_TOP: DMX input folder not found: {SUDS_TOP}")

print(f"Input: {SUDS_TOP}")
print(f"Waveforms: {SEISAN_WAV_DB}")
print(f"Index: {TMP_DIR / '10_waveform_index.csv'}")
print(f"Overwrite existing waveforms: {OVERWRITE}")


In [ ]:
result = convert_dmx_to_seisan(
    rawtop=SUDS_TOP,
    seisan_wav_db=SEISAN_WAV_DB,
    network=NETWORK,
    database=DATABASE,
    sampling_rate=SAMPLING_RATE,
    pattern=PATTERN,
    max_files=MAX_FILES,
    overwrite=OVERWRITE,
    remove_irig=True,
    continue_on_error=True,
    verbose=VERBOSE,
    out_waveform_index=TMP_DIR / "10_waveform_index.csv",
    out_trace_id_map=FAIR_META_DIR / "10_trace_id_mapping.csv",
    out_trace_id_map_tex=FAIR_META_DIR / "10_trace_id_mapping.tex",
    out_trace_id_provenance=FAIR_META_DIR / "10_trace_id_provenance.csv",
    out_failures=TMP_DIR / "10_conversion_failures.csv",
)


In [ ]:
display(result["summary"])
display(result["waveform_index"].head())
display(result["trace_id_map"])
display(result["failures"])


## Review before continuing

The waveform index is read back from saved MiniSEED headers, so its trace count reflects merged output. `status` distinguishes new writes from reused files. Failures include the input path and processing stage; review them before downstream steps. Different inputs that would produce the same filename are reported as collisions.

The two-column mapping contains conversions with saved IDs verified against the proposed IDs, including reused files. The provenance CSV also records attempted mappings for failed conversions. Existing files with different IDs or sample rates are rejected; regenerate them with `OVERWRITE=True`. For datasets with other filename conventions, supply `event_id_func(path)`; for other station conventions, supply `id_fixer(trace, network=...)`.

The old command-line step and shell launcher remain available. Run either that step or this notebook for a conversion run.
